# A6: Naive RAG vs Contextual Retrieval

### Objective

In this assignmen I have implemented a domain specific QA system using:

- Naive RAG
- Contextual Retrieval

I have evaluated both methods using ROUGE scores and compared their performance.

### Imports

In [1]:
import os
import re
import json
from pathlib import Path
from typing import List, Dict

import numpy as np
import torch
import faiss
from tqdm.auto import tqdm
import fitz  # PyMuPDF
print("PyMuPDF working ✅")

PyMuPDF working ✅


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


# Task 1. Source Discovery & Data Preparation

### Task 1.1: Chapter Selection

According to the assignment, the assigned chapter is determined by the last digit of the student ID.

- Student ID: st125999
- Last digit: 9


In [3]:
STUDENT_ID = "st125999"

def assign_chapter(student_id: str) -> int:
    last_digit = int(student_id[-1])
    return 10 if last_digit == 0 else last_digit

CHAPTER_NUM = assign_chapter(STUDENT_ID)

print("Assigned Chapter:", CHAPTER_NUM)

Assigned Chapter: 9


Therefore, this assignment uses **Chapter 9** from the Stanford SLP3 textbook.

### Task 1.2: DOCUMENT PROCESSING



Processing steps:
1. Load raw text from the chapter file
2. Remove formatting noise (page numbers, broken words)
3. Normalize whitespace
4. Split text into meaningful paragraphs


#### Extracting Chapter From Website

In [4]:
import fitz

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""

    for page in doc:
        text += page.get_text()

    return text

raw_text = extract_text("data/9.pdf")

####  Cleaning Text

In [5]:
import re

def clean_text(text):
    # removing unwanted characters
    text = text.replace("\x0c", " ").replace("\u00a0", " ")

    # fixing hyphenated words
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # removing page numbers
    text = re.sub(r"\n\d+\n", "\n", text)

    # merging broken lines
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

    # normalizing spaces 
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

clean_text_data = clean_text(raw_text)

#### Splitting into Paragraphs

In [6]:
# Overlapping chunks improve retrieval performance in RAG systems
def split_into_chunks(text, chunk_size=500, overlap=100):
    
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]

        chunks.append({
            "chunk_id": len(chunks),
            "text": chunk.strip()
        })

        start += chunk_size - overlap

    return chunks

chunks = split_into_chunks(clean_text_data)

print("Total chunks:", len(chunks))

Total chunks: 120


#### Saving Data

In [7]:
import json
from pathlib import Path

OUTPUT_DIR = Path("outputs/task1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_DIR / "chunks.json", "w") as f:
    json.dump(chunks, f, indent=2)

### Task 1.3: QA Pair Generation


A dataset of 20 Question-Answer pairs was constructed based strictly on Chapter 9.

The QA pairs:
- Cover key topics such as instruction tuning, preference alignment, reward models, DPO, and chain-of-thought prompting
- Use only information from the chapter (no external knowledge)
- Are designed to evaluate retrieval and generation quality in later tasks

In [8]:
qa_pairs = [
{"question": "Why are pretrained language models not sufficient for following instructions?",
 "answer": "Because they are trained only to predict the next word, they may ignore the intent of instructions and instead generate contextually similar continuations."},

{"question": "What problem is illustrated by early GPT failures?",
 "answer": "Early GPT models failed to follow instructions and instead produced outputs that continued the prompt without addressing the requested task."},

{"question": "What is instruction tuning?",
 "answer": "Instruction tuning is a method where a pretrained language model is finetuned on a dataset of instructions paired with their correct responses."},

{"question": "What is another name for instruction tuning?",
 "answer": "Instruction tuning is also referred to as supervised fine-tuning (SFT)."},

{"question": "What objective is used during instruction tuning?",
 "answer": "It uses the standard next-token prediction objective with cross-entropy loss, similar to pretraining."},

{"question": "What is a base model?",
 "answer": "A base model is a pretrained language model that has not yet been aligned using instruction tuning or preference-based methods."},

{"question": "What is model alignment?",
 "answer": "Model alignment refers to techniques that adjust language models to better match human needs, such as being helpful and non-harmful."},

{"question": "Why can language models be harmful?",
 "answer": "They can generate unsafe, false, or toxic outputs because their pretraining objective is not aligned with human safety and correctness requirements."},

{"question": "What is preference alignment?",
 "answer": "Preference alignment is a method that improves model outputs by training them based on human preferences over alternative responses."},

{"question": "What is RLHF?",
 "answer": "Reinforcement Learning from Human Feedback is a technique that uses human preference data to fine-tune language models."},

{"question": "What is a reward model?",
 "answer": "A reward model is trained to assign scalar scores to outputs based on how well they align with human preferences."},

{"question": "What does the Bradley-Terry model represent?",
 "answer": "It represents the probability that one output is preferred over another using the difference between their assigned scores."},

{"question": "What is the role of the sigmoid function in preference modeling?",
 "answer": "The sigmoid function converts the difference in scores between outputs into a probability representing preference."},

{"question": "What loss function is used to train reward models?",
 "answer": "Binary cross-entropy loss is used to train reward models based on preference pairs."},

{"question": "What is Direct Preference Optimization (DPO)?",
 "answer": "DPO is a method that directly optimizes language models using preference data without requiring a separate reward model."},

{"question": "What is an advantage of DPO over PPO?",
 "answer": "DPO avoids the need for a separate reward model and does not require expensive sampling during training."},

{"question": "What is test-time compute?",
 "answer": "Test-time compute refers to additional computations performed during inference, such as techniques that improve output quality without retraining the model."},

{"question": "What is chain-of-thought prompting?",
 "answer": "Chain-of-thought prompting is a technique that encourages language models to generate intermediate reasoning steps before producing a final answer."},

{"question": "Why does chain-of-thought prompting improve performance?",
 "answer": "It improves performance by enabling the model to break complex problems into step-by-step reasoning processes."},

{"question": "What is the goal of evaluating instruction-tuned models?",
 "answer": "The goal is to measure how well the model generalizes to new, unseen tasks beyond those used during training."}
]

### Saving QA

In [9]:
with open(OUTPUT_DIR / "qa_pairs.json", "w") as f:
    json.dump(qa_pairs, f, indent=2)

ground_truth = [
    {"question": q["question"], "ground_truth_answer": q["answer"]}
    for q in qa_pairs
]

with open(OUTPUT_DIR / "ground_truth.json", "w") as f:
    json.dump(ground_truth, f, indent=2)

# Task 2. Technique Comparison: Naive RAG vs. Contextual Retrieval

In [10]:
import json
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

import torch
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from rouge_score import rouge_scorer

##### Models Used

- Retriever Model: sentence-transformers/all-MiniLM-L6-v2  
- Generator Model: Qwen/Qwen2.5-3B-Instruct  

### Loading Data

In [11]:
import json
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

import torch
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from rouge_score import rouge_scorer

# CONFIG
STUDENT_ID = "st125999"
CHAPTER_NUM = 9

BASE_DIR = Path(".")
TASK1_DIR = BASE_DIR / "outputs/task1"
TASK2_DIR = BASE_DIR / "outputs/task2"
TASK2_DIR.mkdir(parents=True, exist_ok=True)

CHUNKS_PATH = TASK1_DIR / "chunks.json"
QA_PATH = TASK1_DIR / "ground_truth.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"

TOP_K = 3

# LOAD DATA
with open(CHUNKS_PATH) as f:
    chunks = json.load(f)

with open(QA_PATH) as f:
    qa_data = json.load(f)

print("Chunks:", len(chunks))
print("QA pairs:", len(qa_data))

Chunks: 120
QA pairs: 20


## 2.1 NAIVE RAG

In this section, a standard Retrieval-Augmented Generation (RAG) pipeline was implemented using a basic chunking strategy and vector-based retrieval.

#### Chunking Strategy:
The cleaned chapter text was divided into paragraph-level chunks obtained from Task 1. Each paragraph was treated as an individual retrieval unit.

#### Retriever Model:
The embedding model used was: sentence-transformers/all-MiniLM-L6-v2

Each chunk was encoded into dense vector embeddings and indexed using a FAISS inner-product index (cosine similarity).

#### Retrieval:
For each query, the top-3 most relevant chunks were retrieved using similarity search.

The generation model used was: Qwen/Qwen2.5-3B-Instruct

The retrieved chunks were concatenated into a context, and the model generated answers constrained strictly to the provided context.

#### Loading Models

In [12]:
# Embedding model (Retriever)
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)

# Generator model
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto"
)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

/home/jupyter-st125999/.local/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

### Embeddings and FAISS Index

In [13]:
def embed(texts):
    return embedder.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

chunk_texts = [c["text"] for c in chunks]
embeddings = embed(chunk_texts)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

### Retrieval Function

In [14]:
def retrieve(query, top_k=3):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q_emb, top_k)

    results = []
    for rank, (i, s) in enumerate(zip(idxs[0], scores[0]), start=1):
        c = chunks[int(i)]
        results.append({
            "rank": rank,
            "score": float(s),
            "chunk_id": c["chunk_id"],
            "text": c["text"]
        })

    return results

### Answer Generation

In [15]:
def generate_answer(question, retrieved_chunks):
    context = "\n\n".join([c["text"] for c in retrieved_chunks])

    prompt = f"""
You are answering a question using ONLY the given context.

Rules:
- Do NOT use outside knowledge
- Answer in 1–3 sentences
- If the answer is not clearly in the context, say "Not found in context"

Question:
{question}

Context:
{context}

Answer:
"""

    messages = [
        {"role": "system", "content": "You are a careful academic assistant."},
        {"role": "user", "content": prompt}
    ]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = generator(
        input_text,
        max_new_tokens=150,
        temperature=0.2,
        return_full_text=False
    )[0]["generated_text"]

    return output.strip()

### Naive RAG

In [16]:
naive_outputs = []

for item in tqdm(qa_data, desc="Naive RAG"):
    retrieved = retrieve(item["question"], TOP_K)
    ans = generate_answer(item["question"], retrieved)

    naive_outputs.append({
        "question": item["question"],
        "ground_truth_answer": item["ground_truth_answer"],
        "naive_rag_answer": ans,
        "retrieved_chunks": retrieved   # ✅ IMPORTANT (Task 3 ready)
    })

Naive RAG:   0%|          | 0/20 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


## 2.2 Contextual Retrieval Implementation

To improve retrieval quality, Contextual Retrieval was implemented by enriching each chunk with additional context derived from the full document.

#### Contextual Enrichment:
For each chunk, a short 1–2 sentence explanation was generated using the LLM. This explanation describes how the chunk relates to the full document.

#### Prompt Strategy:

The model was given: The document title (Chapter 9) | A truncated version of the full document| The target chunk

It generated a contextual prefix in the format: This chunk from Chapter 9 discusses..

#### Chunk Transformation:

* Generated Context

* Original Chunk Text

#### Re-Embedding and Indexing:
The enriched chunks were embedded again using the same embedding model and indexed using FAISS.

#### Retrieval with Generation:
The same retrieval and generation pipeline as Naive RAG was used, but with contextualized chunks.

### Context Enrichment Function

In [1]:
def enrich_chunk(chunk, full_doc):
    prompt = f"""
Title: Chapter 9

{full_doc[:4000]}

{chunk}

Write 1–2 sentences explaining what this chunk discusses in the document.

Format:
"This chunk from Chapter 9 discusses ..."
"""

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = generator(
        input_text,
        max_new_tokens=100,
        temperature=0.2,
        return_full_text=False
    )[0]["generated_text"]

    return output.strip()

### Contextual Chunks

In [2]:
full_doc = " ".join(chunk_texts)

contextual_chunks = []

for c in tqdm(chunks, desc="Building Contextual Chunks"):
    ctx = enrich_chunk(c["text"], full_doc)

    contextual_chunks.append({
        "chunk_id": c["chunk_id"],
        "context_prefix": ctx,
        "original_text": c["text"],
        "text": ctx + "\n\n" + c["text"]
    })

NameError: name 'chunk_texts' is not defined

### Embedding Contextual Chunks

In [ ]:
ctx_texts = [c["text"] for c in contextual_chunks]
ctx_embeddings = embed(ctx_texts)

ctx_index = faiss.IndexFlatIP(ctx_embeddings.shape[1])
ctx_index.add(ctx_embeddings)




### Contextual Retriever

In [ ]:
def retrieve_ctx(query, top_k=3):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = ctx_index.search(q_emb, top_k)

    results = []
    for rank, (i, s) in enumerate(zip(idxs[0], scores[0]), start=1):
        c = contextual_chunks[int(i)]
        results.append({
            "rank": rank,
            "score": float(s),
            "chunk_id": c["chunk_id"],
            "text": c["text"]
        })

    return results

### Running Contextual RAG

In [ ]:
contextual_outputs = []

for item in tqdm(qa_data, desc="Contextual RAG"):
    retrieved = retrieve_ctx(item["question"], TOP_K)
    ans = generate_answer(item["question"], retrieved)

    contextual_outputs.append({
        "question": item["question"],
        "ground_truth_answer": item["ground_truth_answer"],
        "contextual_retrieval_answer": ans,
        "retrieved_chunks": retrieved
    })

Contextual RAG:   0%|          | 0/20 [00:00<?, ?it/s]

## 2.3 Evaluation (ROUGE Scores)

In [22]:
def compute_rouge(refs, preds):
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True
    )

    r1, r2, rl = [], [], []

    for ref, pred in zip(refs, preds):
        s = scorer.score(ref, pred)
        r1.append(s["rouge1"].fmeasure)
        r2.append(s["rouge2"].fmeasure)
        rl.append(s["rougeL"].fmeasure)

    return np.mean(r1), np.mean(r2), np.mean(rl)

refs = [x["ground_truth_answer"] for x in qa_data]

naive_preds = [x["naive_rag_answer"] for x in naive_outputs]
ctx_preds = [x["contextual_retrieval_answer"] for x in contextual_outputs]

naive_scores = compute_rouge(refs, naive_preds)
ctx_scores = compute_rouge(refs, ctx_preds)

## 2.4 Results and Analysis

The performance of both methods was evaluated using 20 question-answer pairs generated in Task 1.

Evaluation Metric:
ROUGE scores were used to compare generated answers against ground truth:

* ROUGE-1

* ROUGE-2

* ROUGE-L

Procedure:

Run Naive RAG for all 20 questions,run Contextual Retrieval for all 20 questions,compute average ROUGE scores for both methods

In [23]:
print("\nEvaluation Table:")
print("Method\t\tROUGE-1\tROUGE-2\tROUGE-L")

print(f"Naive RAG\t{naive_scores[0]:.4f}\t{naive_scores[1]:.4f}\t{naive_scores[2]:.4f}")
print(f"Contextual\t{ctx_scores[0]:.4f}\t{ctx_scores[1]:.4f}\t{ctx_scores[2]:.4f}")


Evaluation Table:
Method		ROUGE-1	ROUGE-2	ROUGE-L
Naive RAG	0.3541	0.1665	0.3065
Contextual	0.4052	0.2049	0.3380


The results show that the Contextual Retrieval method outperforms the Naive RAG approach across all evaluation metrics (ROUGE-1, ROUGE-2, and ROUGE-L).

Specifically, ROUGE-1 improved from 0.3541 to 0.4052, ROUGE-2 improved from 0.1665 to 0.2049, and ROUGE-L improved from 0.3065 to 0.3380. This indicates that contextual retrieval produces answers with better lexical overlap, improved phrase-level matching, and more coherent sequence alignment with the ground truth answers.

The improvement occurs because contextual retrieval enriches each chunk with additional semantic information that explains its role within the full document. This added context helps the retriever select more relevant chunks and provides the generator with clearer, more informative input.

As a result, the generated answers are more accurate and aligned with the expected responses compared to the naive RAG approach, which relies only on raw chunk text without contextual augmentation.

In [24]:
def merge_final_outputs(naive_outputs, contextual_outputs):
    final = []

    for n, c in zip(naive_outputs, contextual_outputs):
        final.append({
            "question": n["question"],
            "ground_truth_answer": n["ground_truth_answer"],
            "naive_rag_answer": n["naive_rag_answer"],
            "contextual_retrieval_answer": c["contextual_retrieval_answer"]
        })

    return final


final_response_data = merge_final_outputs(naive_outputs, contextual_outputs)

## Saving Final Output

In [25]:
from pathlib import Path
import json

TASK2_DIR = Path("outputs") / "task2"
TASK2_DIR.mkdir(parents=True, exist_ok=True)
print("Saving to:", TASK2_DIR.resolve())

# Saving contextual chunks
with open(TASK2_DIR / "contextual_chunks.json", "w", encoding="utf-8") as f:
    json.dump(contextual_chunks, f, indent=2, ensure_ascii=False)

print("contextual_chunks.json saved ✅")

Saving to: /home/jupyter-st125999/A6/outputs/task2
contextual_chunks.json saved ✅
